# 01 — Cohort A — import, assess, rank

| | |
|---|---|
| **runs at** | cohort A |
| **reads** | abundance.csv, feature_metadata.txt, sample-metadata.csv |
| **writes** | `cohort-A.csv`, `cohort-A_meta.csv`, `cohort-A_features.csv` |

Cohort A is **batch A** of the SLE data. Batch is the mechanism; cohort is
the entity. This notebook is what one institution runs on its own data, and it
never sees cohort B's rows at any point.

Its counterpart is `02`, which is the
same notebook for the other cohort. They are duplicated rather than looped
because in a real deployment they run on different machines.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository. Download it from Zenodo
# (doi:10.5281/zenodo.20342569), unpack it, and point SLE at the folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 80        # 80 objects keeps p < n in BOTH cohorts
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
else:
    # A stand-in of the same shape, so every cell runs without the download:
    # two batches of unequal size, a case/control split, and a few proteins
    # measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    FEATURE_MAP = 'feature_metadata.txt'
print(f"{'SLE data' if REAL else 'stand-in'}: n_boot={NBOOT}, top_variable={TOPVAR}")

In [ ]:
def build_stand_in():
    """Only used when the Zenodo download is absent."""
    rng = np.random.default_rng(0)
    n, p = 90, 30
    ids = [f'S{i:03d}' for i in range(n)]
    drivers = rng.normal(size=(n, 5))
    X = np.exp(rng.normal(3, 1, size=(1, p)) + drivers @ rng.normal(size=(5, p))
               + rng.normal(scale=0.3, size=(n, p)))
    seqs = [f'seq.{1000+j}.{j%7}' for j in range(p)]
    ab = pd.DataFrame(X, index=pd.Index(ids, name='SampleId'), columns=seqs)
    batch = np.where(np.arange(n) % 3 == 0, 'B', 'A')      # unequal, like the real data
    ab.loc[batch == 'B'] *= 1.6                            # a real batch shift
    meta = pd.DataFrame({
        'DonorId': ids, 'Included_in_study': 'Included', 'Batch': batch,
        'Group': np.where(rng.random(n) < 0.25, 'HV', 'SLE'),
        'Sex': rng.choice(['F', 'M'], n, p=[0.85, 0.15]),
        'Age_group': rng.choice(['26-30', '31-35', '36-40', '41-45'], n),
        'Disease_activity': rng.choice(['Remission', 'LDA', 'MDA', 'HDA'], n),
        'SLEDAI_2K': rng.integers(0, 14, n)}, index=pd.Index(ids, name='SampleId'))
    gene = [f'G{j:02d}' for j in range(p)]
    for a, b in [(1, 2), (10, 11)]:
        gene[b] = gene[a]
    fm = pd.DataFrame({'SeqId': seqs, 'TargetFullName': gene, 'GeneSymbol': gene})
    dup = fm['GeneSymbol'].duplicated(keep=False)
    fm.loc[dup, 'GeneSymbol'] = fm.loc[dup, 'GeneSymbol'] + '_' + fm.loc[dup, 'SeqId']
    fm.to_csv('feature_metadata.txt', sep='\t', index=False)
    return ab, meta

## Step 1 — import, one call per file

Three files, three calls, nothing hidden:

- `abundance.csv` — the measurements, samples by reagents
- `sample-metadata.csv` — who each sample is
- `feature_metadata.txt` — what each reagent measures

In [ ]:
if REAL:
    abundance = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId')
    metadata  = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    features  = pd.read_csv(SLE / 'feature_metadata.txt', sep='\t')
else:
    abundance, metadata = build_stand_in()
    features = pd.read_csv('feature_metadata.txt', sep='\t')

print(f'abundance {abundance.shape[0]} samples x {abundance.shape[1]} reagents')
print(f'metadata  {metadata.shape[0]} samples x {metadata.shape[1]} columns')
print(f'features  {len(features)} reagents described')

## Step 1b — does the data have problems?

Check before you model, not after. Each of these has bitten this project:

- **Counts that disagree between files.** In the related oadr-cpep study, one
  cohort's roster said 83 subjects, its C-peptide file 79, and its autoantibody
  file 72. The model silently used 72 and a slide reported 75.
- **Samples with measurements but no metadata**, or the reverse.
- **Constant features**, which have zero variance and no correlation, so they
  produce non-finite distances and get silently dropped from every bootstrap.
- **Entirely missing features.**

In [ ]:
problems = []
only_ab = set(abundance.index) - set(metadata.index)
only_md = set(metadata.index) - set(abundance.index)
if only_ab: problems.append(f'{len(only_ab)} samples measured but not described')
if only_md: problems.append(f'{len(only_md)} samples described but not measured')

const = abundance.columns[abundance.nunique() <= 1]
if len(const): problems.append(f'{len(const)} constant reagents')
allna = abundance.columns[abundance.isna().all()]
if len(allna): problems.append(f'{len(allna)} entirely missing reagents')
unnamed = set(abundance.columns) - set(features.iloc[:, 0])
if unnamed: problems.append(f'{len(unnamed)} reagents with no entry in feature_metadata')

print('\n'.join(f'  ! {p}' for p in problems) if problems else '  no problems found')
print(f'\nmissing values overall: {100 * abundance.isna().mean().mean():.2f}%')

## Step 1c — take only cohort A's rows

Everything from here uses this cohort's samples only. Excluded subjects are
dropped first, because a subject the study excluded should not reach a model.

In [ ]:
if REAL:
    metadata = metadata[metadata['Included_in_study'] == 'Included']
mine = metadata.index[metadata['Batch'] == 'A']
mine = [s for s in mine if s in abundance.index]

cohort = abundance.loc[mine]
cohort_meta = metadata.loc[mine].copy()
cohort_meta['SLEDAI_band'] = pd.cut(
    pd.to_numeric(cohort_meta.get('SLEDAI_2K'), errors='coerce'),
    [-0.1, 0, 4, 8, 30], labels=['0', '1-4', '5-8', '9+'])
cohort_meta = cohort_meta.astype({'SLEDAI_band': str}).replace('nan', 'NA').fillna('NA')

cohort.to_csv('cohort-A.csv')
cohort_meta.to_csv('cohort-A_meta.csv')
print(f'cohort A: {len(mine)} samples')
print(cohort_meta['Group'].value_counts().to_string())

## Step 2 — rank the features

**Why not LASSO.** oadr-cpep selected features with LASSO, which shrinks the
coefficients of an outcome model to zero. Clustering has no outcome. Nothing is
being predicted, so there is no coefficient to shrink, and LASSO simply does not
apply. Selection here has to be unsupervised.

**What we do instead.** Rank by variance and keep the top N. A feature that
barely varies cannot separate samples, whatever else is true of it.

**Two things that make this fragile, and how they are handled.**

1. Ranking must come **after** batch correction, or you rank the batch shift.
   The command below does log2, then correction, then the ranking, in that order.
2. Clustering cost grows with the **square** of the object count, so this is also
   what makes the run finish. 80 objects here keeps `p < n` in both cohorts,
   which is the privacy rule the aggregation step depends on.

**The principled alternative** is `VarSelLCM`, which selects the variables that
discriminate the latent classes rather than the ones that merely vary. It is an R
package with no Python port, so it is not available here.

In [ ]:
run(['pvclust-py', 'project-features', '--project', 'cohort-A',
     '--matrix', 'cohort-A.csv', '--log2',
     '--metadata', 'cohort-A_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--top-variable', str(TOPVAR),
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

Batch correction is a **no-op here**: this cohort is a single batch, so there is
nothing to correct against. The flag stays because a real cohort usually spans
several plates, and `00_lesson_combat` is where the correction is actually shown.

In [ ]:
feat = pd.read_csv('cohort-A_features.csv')
print(f'{len(feat)} features ranked; the 10 most variable:')
print(feat.head(10)[['feature', 'n_present', 'mean', 'sd']].to_string(index=False))

---
**Next:** `03_aggregate_shared_features` takes this cohort's list and cohort
B's and intersects them. Nothing but the feature list leaves here.